### Pré-processamento do BETA dataset

Nesta etapa, os dados do participante S001 são organizados no formato que o MNE espera, `(n_epochs, n_channels, n_times)`, e o objeto `info` é criado.

In [ ]:
# Carregar os dados

import numpy as np

data = np.load('S001_data.npy')

channels_index, time_points, electrode_index, block_index, target_index = data.shape
channels = ['POz', 'PO3', 'PO4', 'PO5', 'PO6', 'Oz', 'O1', 'O2']
fs = 250

**Monta o array data `[canal, tempo, eletrodo, bloco, alvo]` na forma esperada pelo MNE [n_epochs, n_channels, n_times]**

Para isso, cada par (bloco, alvo) corresponde a uma epoch no formato [8, 710]. Como cada conjunto tem 10 blocos e 12 alvos, totalizam 120 epochs. Os tipos de eletrodos `dry` e `wet` foram separados em dois conjuntos, por serem duas toucas diferentes. Por fim, os labels guardam o índice do alvo de 0 a 11 de cada epoch.

In [ ]:
import mne

epochs_por_eletrodo = []
labels_por_eletrodo = []

for e in range(electrode_index):
    epochs_eletrodo = []
    labels_eletrodo = []

    for b in range(block_index):
        for t in range(target_index):
            epochs_eletrodo.append(data[:, :, e, b, t])
            labels_eletrodo.append(t)

    epochs_por_eletrodo.append(np.array(epochs_eletrodo))
    labels_por_eletrodo.append(np.array(labels_eletrodo))

epochs_dry = epochs_por_eletrodo[0]
labels_dry = labels_por_eletrodo[0]

epochs_wet = epochs_por_eletrodo[1]
labels_wet = labels_por_eletrodo[1]

print('Dry:', epochs_dry.shape, labels_dry.shape)
print('Wet:', epochs_wet.shape, labels_wet.shape)

**Verificação**

Como cada época deve ser idêntica ao trecho correspondente do array original, o trecho abaixo compara a época 0 (bloco 0, alvo 0) e a época 13 (bloco 1, alvo 1). Como os loops percorrem os blocos e, dentro de cada bloco, os alvos, a época de índice `b * 12 + t` corresponde a `(bloco b, alvo t)`.

In [ ]:
print(np.array_equal(epochs_dry[0], data[:, :, 0, 0, 0]))
print(np.array_equal(epochs_dry[13], data[:, :, 0, 1, 1]))
print(labels_dry[0:24])

**Criar o objeto info**

O MNE assume que o sinal está volts, mas o Dataset Beta utiliza unidade em microvolts, então é realizada a conversão ao criar o EpochsArray.

In [ ]:
sfreq = fs
ch_names = channels
ch_types = []

for i in range(channels_index):
    ch_types.append('eeg')

info = mne.create_info(ch_names, sfreq=sfreq, ch_types='eeg')
info

In [ ]:
print('Canais:', info['ch_names'])
print('Frequência de amostragem:', info['sfreq'])
print('Número de canais:', info['nchan'])